# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [6]:
%idle_timeout 2880
%glue_version 4.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

You are already connected to a glueetl session d1c60c76-90cf-4003-b015-9ce65e06b528.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Current idle_timeout is 2880 minutes.
idle_timeout has been set to 2880 minutes.


You are already connected to a glueetl session d1c60c76-90cf-4003-b015-9ce65e06b528.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Setting Glue version to: 4.0


You are already connected to a glueetl session d1c60c76-90cf-4003-b015-9ce65e06b528.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous worker type: G.1X
Setting new worker type to: G.1X


You are already connected to a glueetl session d1c60c76-90cf-4003-b015-9ce65e06b528.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous number of workers: 5
Setting new number of workers to: 5



## Leyendo la Data RAW

In [11]:
DB_RAW = "final_db_bronze"
TABLE_RAW = "raw"

dyf = glueContext.create_dynamic_frame.from_catalog(
    database=DB_RAW,
    table_name=TABLE_RAW
)
df = dyf.toDF()

df.printSchema()
print("Filas:", df.count(), "Columnas:", len(df.columns))

root
 |-- id: string (nullable = true)
 |-- source: string (nullable = true)
 |-- severity: long (nullable = true)
 |-- start_time: string (nullable = true)
 |-- end_time: string (nullable = true)
 |-- start_lat: double (nullable = true)
 |-- start_lng: double (nullable = true)
 |-- end_lat: double (nullable = true)
 |-- end_lng: double (nullable = true)
 |-- distance(mi): double (nullable = true)
 |-- description: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- county: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zipcode: string (nullable = true)
 |-- country: string (nullable = true)
 |-- timezone: string (nullable = true)
 |-- airport_code: string (nullable = true)
 |-- weather_timestamp: string (nullable = true)
 |-- temperature(f): double (nullable = true)
 |-- wind_chill(f): double (nullable = true)
 |-- humidity(%): double (nullable = true)
 |-- pressure(in): double (nullable = true)
 |-- visibility(mi

In [12]:
cols = ["id","severity","start_time","state","city","distance(mi)","weather_timestamp"]
df_small = df.select(*[c for c in cols if c in df.columns])

df_small.show(10, truncate=True)


+---------+--------+--------------------+-----+------------+------------+-------------------+
|       id|severity|          start_time|state|        city|distance(mi)|  weather_timestamp|
+---------+--------+--------------------+-----+------------+------------+-------------------+
|A-2047758|       2| 2019-06-12 10:10:56|   LA|     Zachary|         0.0|2019-06-12 09:53:00|
|A-4694324|       2|2022-12-03 23:37:...|   VA|    Sterling|       0.056|2022-12-03 23:52:00|
|A-5006183|       2|2022-08-20 13:13:...|   CA|      Lompoc|       0.022|2022-08-20 12:56:00|
|A-4237356|       2| 2022-02-21 17:43:04|   MN|      Austin|       1.054|2022-02-21 17:35:00|
|A-6690583|       2| 2020-12-04 01:46:00|   CA| Bakersfield|       0.046|2020-12-04 01:54:00|
|A-1101469|       2| 2021-03-29 07:03:58|   MA|     Peabody|         0.0|2021-03-29 06:53:00|
|A-7222249|       2| 2020-01-14 16:49:23|   OR|   Gold Hill|         0.0|2020-01-14 16:53:00|
|A-6198239|       2|2021-08-13 16:48:...|   FL| Panama City|

### Nulos y Blancos por columna

In [13]:
from pyspark.sql.functions import col, count, when, trim
from pyspark.sql import functions as F

total_rows = df.count()

nulls_df = df.select([
    count(when(col(c).isNull() | (trim(col(c).cast("string")) == ""), c)).alias(c)
    for c in df.columns
])

# Pasar a formato largo y ordenar (Top 25 columnas con más nulos/blancos)
nulls_long = nulls_df.select(
    F.explode(
        F.array(*[
            F.struct(F.lit(c).alias("columna"), F.col(c).alias("nulos_blancos"))
            for c in df.columns
        ])
    ).alias("kv")
).select("kv.columna", "kv.nulos_blancos") \
 .withColumn("pct", F.round((F.col("nulos_blancos")/F.lit(total_rows))*100, 2)) \
 .orderBy(F.col("nulos_blancos").desc())

nulls_long.show(25, truncate=False)


+---------------------+-------------+-----+
|columna              |nulos_blancos|pct  |
+---------------------+-------------+-----+
|end_lng              |220377       |44.08|
|end_lat              |220377       |44.08|
|precipitation(in)    |142616       |28.52|
|wind_chill(f)        |129017       |25.8 |
|wind_speed(mph)      |36987        |7.4  |
|visibility(mi)       |11291        |2.26 |
|wind_direction       |11197        |2.24 |
|humidity(%)          |11130        |2.23 |
|weather_condition    |11101        |2.22 |
|temperature(f)       |10466        |2.09 |
|pressure(in)         |8928         |1.79 |
|weather_timestamp    |7674         |1.53 |
|nautical_twilight    |1483         |0.3  |
|sunrise_sunset       |1483         |0.3  |
|civil_twilight       |1483         |0.3  |
|astronomical_twilight|1483         |0.3  |
|airport_code         |1446         |0.29 |
|street               |691          |0.14 |
|timezone             |507          |0.1  |
|zipcode              |116      

### Duplicados

In [14]:
if "id" in df.columns:
    dups = df.count() - df.dropDuplicates(["id"]).count()
    print("Duplicados por id:", dups)
else:
    print("No existe columna 'id' para validar duplicados.")

Duplicados por id: 0


### Buscando Nulls

In [15]:
tokens = ["null", "NULL", "na", "NA", "n/a", "N/A", "none", "None"]

token_counts = {}
for c in df.columns:
    token_counts[c] = df.filter(trim(col(c).cast("string")).isin(tokens)).count()

# Mostrar las 15 columnas con más tokens tipo "null/NA"
sorted_tokens = sorted(token_counts.items(), key=lambda x: x[1], reverse=True)[:15]
sorted_tokens


[('id', 0), ('source', 0), ('severity', 0), ('start_time', 0), ('end_time', 0), ('start_lat', 0), ('start_lng', 0), ('end_lat', 0), ('end_lng', 0), ('distance(mi)', 0), ('description', 0), ('street', 0), ('city', 0), ('county', 0), ('state', 0)]


### Segmentaciones

### Top 10 Accidentes por Estado

In [16]:
from pyspark.sql.functions import col, count

acc_by_state = (
    df.filter(col("state").isNotNull())
      .groupBy("state")
      .agg(count("*").alias("accidentes"))
      .orderBy(col("accidentes").desc())
      .limit(10)
)

acc_by_state.show(truncate=False)

+-----+----------+
|state|accidentes|
+-----+----------+
|CA   |113274    |
|FL   |56710     |
|TX   |37355     |
|SC   |24737     |
|NY   |22594     |
|NC   |21750     |
|VA   |19515     |
|PA   |19351     |
|MN   |12333     |
|OR   |11559     |
+-----+----------+


In [17]:
pdf_state = acc_by_state.toPandas()

import matplotlib.pyplot as plt
plt.figure()
plt.bar(pdf_state["state"], pdf_state["accidentes"])
plt.title("Top 10 Estados con más accidentes")
plt.xlabel("Estado")
plt.ylabel("Cantidad de accidentes")
plt.show()


### Top 10 Accidentes por Ciudad

In [18]:
acc_by_city = (
    df.filter(col("city").isNotNull())
      .groupBy("city")
      .agg(count("*").alias("accidentes"))
      .orderBy(col("accidentes").desc())
      .limit(10)
)

acc_by_city.show(truncate=False)


+-----------+----------+
|city       |accidentes|
+-----------+----------+
|Miami      |12141     |
|Houston    |11031     |
|Los Angeles|10299     |
|Charlotte  |8979      |
|Dallas     |8245      |
|Orlando    |6985      |
|Austin     |6269      |
|Raleigh    |5553      |
|Nashville  |4689      |
|Baton Rouge|4625      |
+-----------+----------+


### Top 10 Accidentes por Zona Horaria

In [19]:
acc_by_tz = (
    df.filter(col("timezone").isNotNull())
      .groupBy("timezone")
      .agg(count("*").alias("accidentes"))
      .orderBy(col("accidentes").desc())
)

acc_by_tz.show(truncate=False)


+-----------+----------+
|timezone   |accidentes|
+-----------+----------+
|US/Eastern |231397    |
|US/Pacific |133987    |
|US/Central |106012    |
|US/Mountain|28097     |
|           |507       |
+-----------+----------+


# Conclusión del EDA – Dataset US Accidents (RAW)

## Cobertura y volumen

- El dataset presenta alta concentración geográfica de accidentes en pocos estados: CA, FL, TX y SC lideran claramente.
- A nivel ciudad, los accidentes se concentran en grandes áreas urbanas (Miami, Houston, Los Angeles), lo que indica un fuerte sesgo urbano y de densidad poblacional.
- Por zona horaria, la mayor incidencia ocurre en US/Eastern y US/Central, coherente con la distribución demográfica y del tráfico en EE. UU.

## Calidad de Datos
- La ubicación principal (state, city, timezone, severity) tiene casi nulos inexistentes, lo que habilita segmentaciones confiables.
- Variables de clima clave (temperature, humidity, visibility, pressure) presentan bajos porcentajes de nulos (≈2%), por lo que son aptas para análisis.

## Conclusión general
El dataset presenta buena calidad estructural, con nulos concentrados en variables no críticas, y ofrece una base sólida para análisis BI orientado a priorización geográfica y temporal de accidentes, cumpliendo adecuadamente los objetivos del proyecto.